# Calculate ETCCDI climate indices — CESM2-WACCM

This code calculates the climate indices for CESM2-WACCM experiments (G6-1.5K-SAI, G6-1.5K-HiLLA, and SSP2-4.5 baseline).

- **All scenarios** (G6 and SSP2-4.5) are read from our private S3 bucket `s3://reflective-persistent-prod-large/CESM2-WACCM/`.  
  Daily data is split into 5 × 10-year netCDF files per variable/member.  
  Variable names follow CESM conventions: `TREFHTMX` (tasmax), `TREFHTMN` (tasmin), `PRECT` (pr).

Indices calculated (no percentile-based indices):
- daily maximum temperature (`TREFHTMX`)
- daily minimum temperature (`TREFHTMN`)
- daily precipitation rate (`PRECT`)

SOURCE: https://xclim.readthedocs.io/en/stable/indices.html

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
from os.path import join, expanduser
import xclim.indices
import os
import s3fs
import fsspec
from pathlib import Path
import cftime
import dask

In [2]:
# -------------------------------------------------------
# Configuration
# -------------------------------------------------------

model = 'CESM2-WACCM'

S3_BASE = 's3://reflective-persistent-prod-large/CESM2-WACCM/'

# G6 scenarios on our private S3 bucket.
# Note: path directory uses lowercase 'k' (G6-1.5k-*),
# but filenames use '1p5K' — both mappings are stored here.
G6_SCENARIOS = {
    'G6-1.5K-SAI':   {'path_dir': 'G6-1.5k-SAI',   'file_tag': 'G6-1p5K-SAI'},
    'G6-1.5K-HiLLA': {'path_dir': 'G6-1.5k-HiLLA', 'file_tag': 'G6-1p5K-HiLLA'},
}

# Member label in S3 path (r1/r2/r3) → zero-padded number in filename (001/002/003)
ENSEMBLE_MEMBERS = {'r1': '001', 'r2': '002', 'r3': '003'}

# SSP2-4.5: read from the same private S3 bucket as the G6 scenarios.
# Member label (r6/r7/…) → zero-padded number in filename (006/007/…)
# We use members 6-10 because there is a bug in the daily data for 1-5 (note from Cindy as of Thu 4 Jun).
SSP245_MEMBERS = {'r6': '006', 'r7': '007', 'r8': '008', 'r9': '009', 'r10': '010'}

# S3 output root — same structure as the local out_data/ directory was
OUTPUT_ROOT = f's3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/{model}'

fs = s3fs.S3FileSystem(anon=False)

def save_nc(obj, s3_path):
    """Write a DataArray or Dataset to S3 via a local temp file (h5netcdf engine)."""
    import tempfile
    with tempfile.NamedTemporaryFile(suffix='.nc', delete=False) as f:
        tmp = f.name
    try:
        obj.to_netcdf(tmp, engine='h5netcdf')
        fs.put(tmp, s3_path.replace('s3://', ''))
    finally:
        os.remove(tmp)

# CESM2-WACCM f09_g17 grid: 192 lat × 288 lon
# chunks={'time': 365} → 365 × 192 × 288 × 4 B ≈ 81 MB per chunk,
# efficient for S3 reads and aligned with freq='YS' resampling.
CHUNKS = {'time': 365}

## First calculate fixed threshold and index indices

### Temperature indices

| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| tasmax (Tx) | SU | Summer days | Number of days when Tx $>$ 25$^\circ$C | days per year | Fixed threshold |
| tasmax (Tx) | ID | Ice days | Number of days when Tx $<$ 0$^\circ$C | days per year | Fixed threshold |
| tasmax (Tx) | TXn | Coldest days | Annual minimum daily maximum temperature | $^\circ$C | Fixed Index |
| tasmax (Tx) | TXx | Warmest days | Annual maximum daily maximum temperature | $^\circ$C | Fixed Index |
| tasmin (Tn) | TNn | Coldest night | Annual minimum daily minimum temperature | $^\circ$C | Fixed Index |
| tasmin (Tn) | TNx | Warmest night | Annual maximum daily minimum temperature | days per year | Fixed Index |
| tasmin (Tn) | FD | Frost days | Number of days when Tn < 0$^\circ$C | days per year | Fixed threshold |
| tasmin (Tn) | TN | Tropical nights | Number of days when Tn $>$ 20$^\circ$C | days per year | Fixed threshold |

In [3]:
for scenario, meta in G6_SCENARIOS.items():
    for member, member_num in ENSEMBLE_MEMBERS.items():
        print(f'Processing scenario={scenario}, member={member}')

        aday_path = f'{S3_BASE}{meta["path_dir"]}/{member}/ADAY/'
        file_tag = meta['file_tag']

        tasmax_files = sorted(fs.glob(
            f'{aday_path}b.e21.BW.f09_g17.SSP245-{file_tag}.{member_num}.cam.h1.TREFHTMX.*.nc'
        ))
        tasmin_files = sorted(fs.glob(
            f'{aday_path}b.e21.BW.f09_g17.SSP245-{file_tag}.{member_num}.cam.h1.TREFHTMN.*.nc'
        ))

        tasmax_ds = xr.open_mfdataset(
            [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in tasmax_files],
            combine='nested', concat_dim='time',
            engine='h5netcdf', chunks=CHUNKS,
            coords='minimal', compat='override'
        )
        tasmin_ds = xr.open_mfdataset(
            [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in tasmin_files],
            combine='nested', concat_dim='time',
            engine='h5netcdf', chunks=CHUNKS,
            coords='minimal', compat='override'
        )

        tasmax_ds = tasmax_ds.sel(time=~tasmax_ds.get_index('time').duplicated())
        tasmin_ds = tasmin_ds.sel(time=~tasmin_ds.get_index('time').duplicated())

        tx = tasmax_ds['TREFHTMX']  # units: K
        tn = tasmin_ds['TREFHTMN']  # units: K

        SU  = xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS')
        ID  = xclim.indices.ice_days(tx, thresh='0 degC', freq='YS')
        TXX = xclim.indices.tx_max(tx, freq='YS')
        TXN = xclim.indices.tx_min(tx, freq='YS')
        FD  = xclim.indices.frost_days(tn, freq='YS')
        TN  = xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS')
        TNX = xclim.indices.tn_max(tn, freq='YS')
        TNN = xclim.indices.tn_min(tn, freq='YS')

        su, id_, txx, txn, fd, tn_, tnx, tnn = dask.compute(SU, ID, TXX, TXN, FD, TN, TNX, TNN)

        out_dir = join(OUTPUT_ROOT, scenario, member)

        save_nc(su,  join(out_dir, 'SU.nc'))
        save_nc(id_, join(out_dir, 'ID.nc'))
        save_nc(txx, join(out_dir, 'TXX.nc'))
        save_nc(txn, join(out_dir, 'TXN.nc'))
        save_nc(fd,  join(out_dir, 'FD.nc'))
        save_nc(tn_, join(out_dir, 'TN.nc'))
        save_nc(tnx, join(out_dir, 'TNX.nc'))
        save_nc(tnn, join(out_dir, 'TNN.nc'))

        print(f'  -> Temperature indices saved to {out_dir}')

Processing scenario=G6-1.5K-SAI, member=r1


/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/G6-1.5K-SAI/r1
Processing scenario=G6-1.5K-SAI, member=r2


/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/G6-1.5K-SAI/r2
Processing scenario=G6-1.5K-SAI, member=r3


...

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/G6-1.5K-HiLLA/r1
Processing scenario=G6-1.5K-HiLLA, member=r2


/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/G6-1.5K-HiLLA/r2
Processing scenario=G6-1.5K-HiLLA, member=r3


/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_709/1583512122.py:15: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/G6-1.5K-HiLLA/r3


In [3]:
# SSP2-4.5 temperature indices — from private S3 bucket
for member, member_num in SSP245_MEMBERS.items():
    print(f'Processing scenario=SSP2-4.5, member={member}')

    day_path = f'{S3_BASE}SSP2-4.5/{member}/day/'

    tasmax_files = sorted(fs.glob(
        f'{day_path}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{member_num}.cam.h1.TREFHTMX.*.nc'
    ))
    tasmin_files = sorted(fs.glob(
        f'{day_path}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{member_num}.cam.h1.TREFHTMN.*.nc'
    ))

    tasmax_ds = xr.open_mfdataset(
        [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in tasmax_files],
        combine='nested', concat_dim='time',
        engine='h5netcdf', chunks=CHUNKS,
        coords='minimal', compat='override'
    )
    tasmin_ds = xr.open_mfdataset(
        [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in tasmin_files],
        combine='nested', concat_dim='time',
        engine='h5netcdf', chunks=CHUNKS,
        coords='minimal', compat='override'
    )

    tasmax_ds = tasmax_ds.sel(time=~tasmax_ds.get_index('time').duplicated())
    tasmin_ds = tasmin_ds.sel(time=~tasmin_ds.get_index('time').duplicated())

    tx = tasmax_ds['TREFHTMX']  # units: K
    tn = tasmin_ds['TREFHTMN']  # units: K

    SU  = xclim.indices.tx_days_above(tx, thresh='25.0 degC', freq='YS')
    ID  = xclim.indices.ice_days(tx, thresh='0 degC', freq='YS')
    TXX = xclim.indices.tx_max(tx, freq='YS')
    TXN = xclim.indices.tx_min(tx, freq='YS')
    FD  = xclim.indices.frost_days(tn, freq='YS')
    TN  = xclim.indices.tn_days_above(tn, thresh='20.0 degC', freq='YS')
    TNX = xclim.indices.tn_max(tn, freq='YS')
    TNN = xclim.indices.tn_min(tn, freq='YS')

    su, id_, txx, txn, fd, tn_, tnx, tnn = dask.compute(SU, ID, TXX, TXN, FD, TN, TNX, TNN)

    out_dir = join(OUTPUT_ROOT, 'SSP2-4.5', member)

    save_nc(su,  join(out_dir, 'SU.nc'))
    save_nc(id_, join(out_dir, 'ID.nc'))
    save_nc(txx, join(out_dir, 'TXX.nc'))
    save_nc(txn, join(out_dir, 'TXN.nc'))
    save_nc(fd,  join(out_dir, 'FD.nc'))
    save_nc(tn_, join(out_dir, 'TN.nc'))
    save_nc(tnx, join(out_dir, 'TNX.nc'))
    save_nc(tnn, join(out_dir, 'TNN.nc'))

    print(f'  -> Temperature indices saved to {out_dir}')

Processing scenario=SSP2-4.5, member=r6


/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r6
Processing scenario=SSP2-4.5, member=r7


/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r7
Processing scenario=SSP2-4.5, member=r8


...

/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r9
Processing scenario=SSP2-4.5, member=r10


/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open_mfdataset(
/tmp/ipykernel_136/2394424788.py:14: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  tasmax_ds = xr.open

  -> Temperature indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r10


/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebook/lib/python3.12/site-packages/dask/_task_spec.py:741: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/srv/conda/envs/notebo

### Precipitation indices

In [4]:
# To convert precipitation rate to mm/day
SECONDS_PER_DAY = 86400
M_TO_MM = 1000

def to_mm_day(da, varname=None):
    """
    Convert precipitation rate to mm/day when units look like:
      - kg m-2 s-1  (water flux; 1 kg/m^2 = 1 mm water)
      - m s-1       (water-equivalent depth rate)
    Otherwise return da unchanged.
    """
    units = (da.attrs.get("units") or "").strip().lower().replace(" ", "")

    is_kg_m2_s = units in {"kgm-2s-1", "kg/m^2/s", "kg/m2/s", "kgm^-2s^-1", "kgm**-2s**-1"}
    is_m_s     = units in {"ms-1", "m/s", "m s-1", "m/s-1"} or units == "ms^-1"

    if is_kg_m2_s:
        out = da * SECONDS_PER_DAY  # kg/m^2/day == mm/day
    elif is_m_s:
        out = da * SECONDS_PER_DAY * M_TO_MM
    else:
        if varname is not None and varname.upper() in {"PRECT", "PRECC", "PRECL", "PR"}:
            out = da * SECONDS_PER_DAY
        else:
            return da

    out = out.assign_attrs(da.attrs)
    out.attrs["units"] = "mm/day"
    return out


| Variable Used | Index | Name | Definition | Unit | Type |
|---|---|---|---|---|---|
| pr (PR) | PRCPTOT | Total rainfall | Annual sum of precipitation | mm | Fixed Index |
| pr (PR) | SDII | Simple daily intensity | Mean precipitation falling on days when PR $\geq$ 1 mm | mm | Fixed Index |
| pr (PR) | Rx1day | Wettest day | Annual maximum precipitation in a single day | mm | Fixed Index |
| pr (PR) | Rx5day | Wettest pentad | Annual maximum precipitation falling on 5 consecutive days | mm | Fixed Index |
| pr (PR) | CDD | Consecutive dry days | Longest spell of consecutive days when PR $\leq$ 1 mm | days per year | Fixed index/spell |
| pr (PR) | CWD | Consecutive wet days | Longest spell of consecutive days when PR $\geq$ 1 mm | days per year | Fixed index/spell |
| pr (PR) | R10mm | Heavy precipitation days | Number of days when precipitation $\geq$ 10 mm | days per year | Fixed threshold |
| pr (PR) | R20mm | Very heavy precipitation days | Number of days when precipitation $\geq$ 20 mm | days per year | Fixed threshold |

In [ ]:
import gc

for scenario, meta in G6_SCENARIOS.items():
    for member, member_num in ENSEMBLE_MEMBERS.items():
        print(f'Processing scenario={scenario}, member={member}')

        aday_path = f'{S3_BASE}{meta["path_dir"]}/{member}/ADAY/'
        file_tag = meta['file_tag']

        pr_files = sorted(fs.glob(
            f'{aday_path}b.e21.BW.f09_g17.SSP245-{file_tag}.{member_num}.cam.h1.PRECT.*.nc'
        ))

        ds = xr.open_mfdataset(
            [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in pr_files],
            combine='nested', concat_dim='time',
            engine='h5netcdf', chunks=CHUNKS,
            coords='minimal', compat='override'
        )

        ds = ds.sel(time=~ds.get_index('time').duplicated())
        pr = to_mm_day(ds['PRECT'], varname='PRECT')  # CESM units: m/s → mm/day

        out_dir = join(OUTPUT_ROOT, scenario, member)

        # --- Pass 1: simple per-timestep aggregations ---
        prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
            xclim.indices.prcptot(pr, freq='YS'),
            xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS'),
            xclim.indices.max_1day_precipitation_amount(pr, freq='YS'),
            xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS'),
            xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>='),
            xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>='),
        )
        save_nc(prcptot, join(out_dir, 'PRCPTOT.nc')); del prcptot
        save_nc(sdii,    join(out_dir, 'SDII.nc'));    del sdii
        save_nc(rx1d,    join(out_dir, 'RX1D.nc'));    del rx1d
        save_nc(rx5d,    join(out_dir, 'RX5D.nc'));    del rx5d
        save_nc(r10mm,   join(out_dir, 'R10MM.nc'));   del r10mm
        save_nc(r20mm,   join(out_dir, 'R20MM.nc'));   del r20mm
        gc.collect()

        # --- Pass 2: consecutive-spell indices (memory-intensive, kept separate) ---
        cdd, cwd = dask.compute(
            xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS'),
            xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS'),
        )
        save_nc(cdd, join(out_dir, 'CDD.nc')); del cdd
        save_nc(cwd, join(out_dir, 'CWD.nc')); del cwd

        del ds, pr
        gc.collect()

        print(f'  -> Precipitation indices saved to {out_dir}')

In [5]:
import gc

# SSP2-4.5 precipitation indices — from private S3 bucket
for member, member_num in SSP245_MEMBERS.items():
    print(f'Processing scenario=SSP2-4.5, member={member}')

    day_path = f'{S3_BASE}SSP2-4.5/{member}/day/'

    pr_files = sorted(fs.glob(
        f'{day_path}b.e21.BWSSP245cmip6.f09_g17.CMIP6-SSP2-4.5-WACCM.{member_num}.cam.h1.PRECT.*.nc'
    ))

    ds = xr.open_mfdataset(
        [fsspec.open(f's3://{f}', mode='rb', anon=False).open() for f in pr_files],
        combine='nested', concat_dim='time',
        engine='h5netcdf', chunks=CHUNKS,
        coords='minimal', compat='override'
    )

    ds = ds.sel(time=~ds.get_index('time').duplicated())
    pr = to_mm_day(ds['PRECT'], varname='PRECT')  # CESM units: m/s → mm/day

    out_dir = join(OUTPUT_ROOT, 'SSP2-4.5', member)

    # --- Pass 1: simple per-timestep aggregations ---
    prcptot, sdii, rx1d, rx5d, r10mm, r20mm = dask.compute(
        xclim.indices.prcptot(pr, freq='YS'),
        xclim.indices.daily_pr_intensity(pr, thresh='1 mm/day', freq='YS'),
        xclim.indices.max_1day_precipitation_amount(pr, freq='YS'),
        xclim.indices.max_n_day_precipitation_amount(pr, window=5, freq='YS'),
        xclim.indices.wetdays(pr, thresh='10 mm/day', freq='YS', op='>='),
        xclim.indices.wetdays(pr, thresh='20 mm/day', freq='YS', op='>='),
    )
    save_nc(prcptot, join(out_dir, 'PRCPTOT.nc')); del prcptot
    save_nc(sdii,    join(out_dir, 'SDII.nc'));    del sdii
    save_nc(rx1d,    join(out_dir, 'RX1D.nc'));    del rx1d
    save_nc(rx5d,    join(out_dir, 'RX5D.nc'));    del rx5d
    save_nc(r10mm,   join(out_dir, 'R10MM.nc'));   del r10mm
    save_nc(r20mm,   join(out_dir, 'R20MM.nc'));   del r20mm
    gc.collect()

    # --- Pass 2: consecutive-spell indices (memory-intensive, kept separate) ---
    cdd, cwd = dask.compute(
        xclim.indices.maximum_consecutive_dry_days(pr, '1 mm/day', freq='YS'),
        xclim.indices.maximum_consecutive_wet_days(pr, thresh='1 mm/day', freq='YS'),
    )
    save_nc(cdd, join(out_dir, 'CDD.nc')); del cdd
    save_nc(cwd, join(out_dir, 'CWD.nc')); del cwd

    del ds, pr
    gc.collect()

    print(f'  -> Precipitation indices saved to {out_dir}')

Processing scenario=SSP2-4.5, member=r6


/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_1

  -> Precipitation indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r6
Processing scenario=SSP2-4.5, member=r7


/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_1

  -> Precipitation indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r7
Processing scenario=SSP2-4.5, member=r8


...

  -> Precipitation indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r8
Processing scenario=SSP2-4.5, member=r9


/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_1

  -> Precipitation indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r9
Processing scenario=SSP2-4.5, member=r10


/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_136/3497831241.py:13: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 365. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_mfdataset(
/tmp/ipykernel_1

  -> Precipitation indices saved to s3://reflective-persistent-prod/alistairduffey/ETCCDI/ETCCDI_indices_annual/CESM2-WACCM/SSP2-4.5/r10
